# Compilador PyChua
**Léxico → Indentación → Sintáctico → Semántico**

## Sección 0 — Setup e imports

In [ ]:
import sys
import os

# Asegura que los módulos del compilador sean encontrados
RAIZ = os.path.abspath("")
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

from f1_analizador_lexico.analizador_lexico import LexerQuechua, imprimir_tabla
from f1_analizador_lexico.indent_processor import IndentProcessor
from f2_analizador_sintactico.parser import Parser, imprimir_ast
from f3_analizador_semantico.analizador_semantico import AnalizadorSemantico, imprimir_resultado_semantico
from f3_analizador_semantico.tabla_simbolos import imprimir_tabla_simbolos

---
## Sección 1 — Selección del archivo fuente

In [3]:
ARCHIVO = "ejemploAlexis.txt"

ruta = os.path.join(RAIZ, "src", ARCHIVO)
with open(ruta, encoding="utf-8") as f:
    codigo = f.read()

print(f"=== CÓDIGO FUENTE: {ARCHIVO} ===")
print("=" * 50)
print(codigo)

=== CÓDIGO FUENTE: ejemploAlexis.txt ===
@Crear una función que permita entrar a 
@una discoteca si la persona es mayor de edad

ruway entrada(edad):
    ari_chayqa(edad >= 18):
        qillqay("Puedes pasar a la discoteca")
    mana_chayqa:
        qillqay("No puedes pasar")

entrada(22)



---
## Sección 2 — Módulo 1: Analizador Léxico

Tokeniza el código fuente y clasifica cada lexema.

**Entrada:** cadena de texto (`codigo`)  
**Salida:** lista de `Token(tipo, lexema, linea, columna)` + errores léxicos

In [4]:
lexer = LexerQuechua(codigo)
tokens_raw = lexer.analizar()

imprimir_tabla(tokens_raw, lexer.errores)


  ANALIZADOR LÉXICO — LENGUAJE QUECHUA
+-----------------+-------------------------------+--------+--------+
| TIPO DE TOKEN   | LEXEMA                        | LÍNEA  | COLUMNA|
+-----------------+-------------------------------+--------+--------+
| FIN_DE_LINEA    | 
                             | 1      | 41     |
| FIN_DE_LINEA    | 
                             | 2      | 46     |
| FIN_DE_LINEA    | 
                             | 3      | 1      |
| PALABRA_CLAVE   | ruway                         | 4      | 1      |
| IDENTIFICADOR   | entrada                       | 4      | 7      |
| PAREN_ABRE      | (                             | 4      | 14     |
| IDENTIFICADOR   | edad                          | 4      | 15     |
| PAREN_CIERRA    | )                             | 4      | 19     |
| DOS_PUNTOS      | :                             | 4      | 20     |
| FIN_DE_LINEA    | 
                             | 4      | 21     |
| PALABRA_CLAVE   | ari_chayqa                    

---
## Sección 3 — Módulo 2: Procesador de Indentación

Inyecta tokens `INDENT` y `DEDENT` basándose en los niveles de indentación del código fuente.

**Entrada:** `tokens_raw` (salida del léxico)  
**Salida:** `tokens_indent` — misma lista con tokens INDENT/DEDENT insertados

In [5]:
proc = IndentProcessor(tokens_raw)
tokens_indent = proc.procesar()

SEP = "=" * 65
print(SEP)
print("TOKENS DESPUÉS DE PROCESAR INDENTACIÓN")
print(SEP)
print(f"{'TIPO':<32} {'LEXEMA':<22} {'LÍNEA':<8} COL")
print("-" * 65)
for tok in tokens_indent:
    marca = " ◄" if tok.tipo in ("INDENT", "DEDENT") else ""
    print(f"{tok.tipo:<32} {repr(tok.lexema):<22} {tok.linea:<8} {tok.columna}{marca}")

print()
if proc.errores:
    print("ERRORES DE INDENTACIÓN:")
    for e in proc.errores:
        print(" ", e)
else:
    print("Sin errores de indentación.")

TOKENS DESPUÉS DE PROCESAR INDENTACIÓN
TIPO                             LEXEMA                 LÍNEA    COL
-----------------------------------------------------------------
PALABRA_CLAVE                    'ruway'                4        1
IDENTIFICADOR                    'entrada'              4        7
PAREN_ABRE                       '('                    4        14
IDENTIFICADOR                    'edad'                 4        15
PAREN_CIERRA                     ')'                    4        19
DOS_PUNTOS                       ':'                    4        20
FIN_DE_LINEA                     '\n'                   4        21
INDENT                           'INDENT'               5        1 ◄
PALABRA_CLAVE                    'ari_chayqa'           5        5
PAREN_ABRE                       '('                    5        15
IDENTIFICADOR                    'edad'                 5        16
OP_COMPARACION                   '>='                   5        21
LITERAL_ENTE

---
## Sección 4 — Módulo 3: Analizador Sintáctico

Construye el Árbol Sintáctico Abstracto (AST) a partir de la secuencia de tokens.

**Entrada:** `tokens_indent` (salida del procesador de indentación)  
**Salida:** `NodoPrograma` — raíz del AST + errores sintácticos

In [6]:
parser = Parser(tokens_indent)
ast = parser.parsear()

print("=== AST (Árbol Sintáctico Abstracto) ===")
print("=" * 50)
print()
imprimir_ast(ast)

print()
if parser.errores:
    print("ERRORES SINTÁCTICOS:")
    for e in parser.errores:
        print(" ", e)
else:
    print("Sin errores sintácticos.")

=== AST (Árbol Sintáctico Abstracto) ===

Programa
  Funcion 'entrada'  (L4)
    Param 'edad'
    Si  (L5)
      Condicion:
        Binario '>='  (L5)
          Identificador 'edad'  (L5)
          Literal LITERAL_ENTERO = '18'  (L5)
      Entonces:
        Llamada 'qillqay'  (L6)
          Literal LITERAL_CADENA = '"Puedes pasar a la discoteca"'  (L6)
      SiNo:
        Llamada 'qillqay'  (L8)
          Literal LITERAL_CADENA = '"No puedes pasar"'  (L8)
  Llamada 'entrada'  (L10)
    Literal LITERAL_ENTERO = '22'  (L10)

Sin errores sintácticos.


---
## Sección 5 — Módulo 4: Analizador Semántico

Recorre el AST con una tabla de símbolos por ámbitos (global / función / clase) y valida lo que la gramática no puede atrapar: variables no declaradas, funciones o clases redeclaradas, aridad de llamadas, uso de `kutichiy`/`usqhaychiy`/`katiy` fuera de contexto y compatibilidad básica de tipos.

**Entrada:** `ast` (salida del sintáctico)
**Salida:** lista de errores semánticos + tabla de símbolos por ámbito

In [ ]:
analizador = AnalizadorSemantico()
errores_semanticos = analizador.analizar(ast)

imprimir_resultado_semantico(errores_semanticos)
imprimir_tabla_simbolos(analizador.ambitos)